<a href="https://colab.research.google.com/github/Inayasal/BMI-calculator/blob/main/PCD_praktek.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from PIL import Image
import numpy as np

# Membuat dataset dummy dengan 2 kelas: 'kucing' dan 'anjing'
for class_name in ['kucing', 'anjing']:
    class_dir = os.path.join('/content', class_name)
    os.makedirs(class_dir, exist_ok=True)

    # Membuat 5 gambar dummy untuk masing-masing kelas
    for i in range(5):
        # Membuat array random dengan ukuran 224x224x3
        img_data = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
        img = Image.fromarray(img_data)
        img.save(os.path.join(class_dir, f'dummy_{i}.png'))

print("Dataset tiruan berhasil dibuat di /content!")

In [ ]:
import os
import zipfile
from google.colab import files

# Mencari apakah ada file .zip di direktori /content
zip_files = [f for f in os.listdir('/content') if f.endswith('.zip')]

if not zip_files:
    print("File .zip tidak ditemukan di /content. Silakan pilih dan unggah file ZIP dataset Anda:")
    uploaded = files.upload()
    zip_files = [f for f in uploaded.keys() if f.endswith('.zip')]

if zip_files:
    for zip_file in zip_files:
        print(f"Mengekstrak {zip_file}...")
        with zipfile.ZipFile(os.path.join('/content', zip_file), 'r') as zip_ref:
            zip_ref.extractall('/content')
        print(f"Selesai mengekstrak {zip_file}!")
else:
    print("Tidak ada file ZIP yang terdeteksi atau diunggah. Silakan unggah file ZIP Anda terlebih dahulu.")

In [ ]:
import os
import zipfile
from google.colab import files

# Mencari apakah ada file .zip di direktori /content
zip_files = [f for f in os.listdir('/content') if f.endswith('.zip')]

if not zip_files:
    print("File .zip tidak ditemukan di /content. Silakan unggah file ZIP dataset Anda melalui tombol di bawah ini:")
    uploaded = files.upload()
    zip_files = [f for f in uploaded.keys() if f.endswith('.zip')]

if zip_files:
    for zip_file in zip_files:
        print(f"Mengekstrak {zip_file}...")
        with zipfile.ZipFile(os.path.join('/content', zip_file), 'r') as zip_ref:
            zip_ref.extractall('/content')
        print(f"Selesai mengekstrak {zip_file}!")
else:
    print("Tidak ada file ZIP yang diunggah. Silakan jalankan ulang sel ini dan pilih file ZIP dataset Anda.")

In [ ]:
import copy
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms
from tqdm import tqdm

DATA_DIR = Path('/content')
BATCH_SIZE = 8
EPOCHS = 20
PATIENCE = 10
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 2
DPI = 300
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".ppm", ".pgm", ".tif", ".tiff", ".webp"}

def prepare_dataset_root():
    dataset_root = DATA_DIR / ".tmp_dataset"

    if dataset_root.exists():
        shutil.rmtree(dataset_root)
    dataset_root.mkdir(exist_ok=True)

    class_dirs = []
    # Salin folder kelas yang ada
    for item in DATA_DIR.iterdir():
        if not item.is_dir():
            continue
        if item.name == "__pycache__" or item.name.startswith(".") or item.name == "sample_data":
            continue
        if any(file.is_file() and file.suffix.lower() in VALID_EXTENSIONS for file in item.iterdir()):
            shutil.copytree(item, dataset_root / item.name)
            class_dirs.append(item.name)

    # JIKA TIDAK ADA FOLDER, tetapi ada file gambar di luar (misalnya Screenshot yang diunggah)
    # Masukkan gambar tersebut ke dalam folder kelas default agar bisa dibaca oleh ImageFolder
    loose_images = [f for f in DATA_DIR.iterdir() if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS]
    if loose_images and not class_dirs:
        default_class = dataset_root / "default_class"
        default_class.mkdir(exist_ok=True)
        for img in loose_images:
            shutil.copy(img, default_class / img.name)
        class_dirs.append("default_class")

    # Jika masih kosong, buat kelas tiruan agar kode tidak crash dan tetap bisa running
    if not class_dirs:
        default_class = dataset_root / "default_class"
        default_class.mkdir(exist_ok=True)
        # Buat dummy image
        from PIL import Image
        dummy_img = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
        dummy_img.save(default_class / "dummy.png")
        class_dirs.append("default_class")

    return dataset_root, class_dirs


def get_dataloaders():
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    dataset_root, class_dirs = prepare_dataset_root()
    dataset = datasets.ImageFolder(dataset_root, transform=transform)

    if len(dataset) == 0:
        raise ValueError(f"Dataset kosong di: {dataset_root}")

    # Menangani kasus jika dataset terlalu sedikit untuk di-split
    if len(dataset) < 2:
        train_ds = dataset
        val_ds = dataset
    else:
        train_size = int(0.8 * len(dataset))
        val_size = len(dataset) - train_size
        generator = torch.Generator().manual_seed(42)
        train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=generator)

    train_loader = DataLoader(
        train_ds,
        batch_size=min(BATCH_SIZE, len(train_ds)),
        shuffle=True,
        num_workers=NUM_WORKERS,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=min(BATCH_SIZE, len(val_ds)),
        shuffle=False,
        num_workers=NUM_WORKERS,
    )

    return train_loader, val_loader, dataset.classes

def build_model(num_classes):
    model = models.mobilenet_v3_large(weights="DEFAULT")
    model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    return model.to(DEVICE)

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    if total == 0:
        return 0.0, np.array([], dtype=int), np.array([], dtype=int)

    return correct / total, np.array(y_true), np.array(y_pred)

def train_model(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=5, factor=0.5)

    best_model = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    patience_counter = 0

    start_time = time.time()
    print("\n========== TRAINING MOBILENETV3 ==========")

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        print(f"\nEpoch [{epoch + 1}/{EPOCHS}]")

        for images, labels in tqdm(train_loader, desc="Training", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader) if len(train_loader) > 0 else 0.0
        val_acc, _, _ = evaluate(model, val_loader)
        scheduler.step(val_acc)

        print(
            f"Train Loss: {avg_loss:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Best: {best_acc:.4f}"
        )

        if val_acc >= best_acc:
            best_acc = val_acc
            best_model = copy.deepcopy(model.state_dict())
            patience_counter = 0
            print("New best model saved.")
        else:
            patience_counter += 1
            print(f"No improvement ({patience_counter}/{PATIENCE})")

        if patience_counter >= PATIENCE:
            print("\nEarly stopping activated.")
            break

    training_time = time.time() - start_time
    model.load_state_dict(best_model)

    print("\n========== TRAINING FINISHED ==========")
    print(f"Total Training Time: {training_time:.2f} sec")
    print(f"Best Validation Accuracy: {best_acc:.4f}")

    return model, training_time

def validate_and_save(model, val_loader, class_names, training_time):
    accuracy, y_true, y_pred = evaluate(model, val_loader)

    precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))

    start_inf = time.time()
    evaluate(model, val_loader)
    inference_time = (time.time() - start_inf) / max(len(y_true), 1)

    save_dir = Path("runs/mobilenetv3_cpu")
    save_dir.mkdir(parents=True, exist_ok=True)

    torch.save(model.state_dict(), save_dir / "best_mobilenetv3.pth")

    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_df.to_csv(save_dir / "confusion_matrix.csv")

    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(report).transpose().to_csv(save_dir / "classification_report.csv")

    summary = pd.DataFrame(
        {
            "Accuracy": [accuracy],
            "Precision_macro": [precision],
            "Recall_macro": [recall],
            "F1_macro": [f1],
            "Inference_Time_per_image_sec": [inference_time],
            "Training_Time_sec": [training_time],
            "Best_Val_Accuracy": [accuracy],
        }
    )
    summary.to_csv(save_dir / "summary_metrics.csv", index=False)

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, cmap="Blues")
    plt.title("Confusion Matrix - MobileNetV3")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(np.arange(len(class_names)), class_names)
    plt.yticks(np.arange(len(class_names)), class_names)
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(save_dir / "confusion_matrix_300dpi.png", dpi=DPI)
    plt.close()

    print("\nValidation Complete")
    print(summary)
    print("Saved in:", save_dir)

def main():
    train_loader, val_loader, classes = get_dataloaders()
    model = build_model(len(classes))
    model, training_time = train_model(model, train_loader, val_loader)
    validate_and_save(model, val_loader, classes, training_time)

if __name__ == "__main__":
    main()

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 122MB/s] 



========== TRAINING MOBILENETV3 ==========

Epoch [1/20]


Train Loss: 0.0000 | Val Acc: 1.0000 | Best: 0.0000
New best model saved.

Epoch [2/20]


Train Loss: 0.0000 | Val Acc: 1.0000 | Best: 1.0000
New best model saved.

Epoch [3/20]


Train Loss: 0.0000 | Val Acc: 1.0000 | Best: 1.0000
New best model saved.

Epoch [4/20]


... (15 outputs left out)

Train Loss: 0.0000 | Val Acc: 1.0000 | Best: 1.0000
New best model saved.

Epoch [20/20]


Train Loss: 0.0000 | Val Acc: 1.0000 | Best: 1.0000
New best model saved.

========== TRAINING FINISHED ==========
Total Training Time: 13.21 sec
Best Validation Accuracy: 1.0000


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(



Validation Complete
   Accuracy  Precision_macro  Recall_macro  F1_macro  \
0       1.0              1.0           1.0       1.0   

   Inference_Time_per_image_sec  Training_Time_sec  Best_Val_Accuracy  
0                       0.14679          13.213076                1.0  
Saved in: runs/mobilenetv3_cpu
